
# 2. Satellite Data Preprocessing

**Project:** Dream-in-Orbit — An AI That Simulates a Satellite

**Notebook:** 02_Satellite_Data_Preprocessing.ipynb

## 2.1 Introduction

This notebook focuses on preparing satellite telemetry data for artificial intelligence modeling and simulation.

Building on the findings from Notebook 01, the preprocessing workflow will address data integration, temporal consistency, feature preparation, and model readiness.

The preprocessing strategy will preserve meaningful satellite operating patterns while minimizing unnecessary data transformations.

### Objectives

1. Examine the structure and compatibility of the available telemetry datasets.
2. Identify the variables relevant to the satellite simulation objectives.
3. Establish a consistent temporal representation across datasets.
4. Handle data quality issues identified during exploratory analysis.
5. Prepare appropriate features for subsequent AI modeling.
6. Produce reproducible, model-ready datasets.

### Methodology

Preprocessing decisions will be guided by the exploratory findings from Notebook 01.

Previously completed analyses will be referenced rather than unnecessarily repeated.

Special attention will be given to preserving temporal relationships, avoiding data leakage, and retaining potentially meaningful satellite operating conditions.



## 2.2 Satellite Dataset Structure and Integration

### 2.2.1 Identification of Available Telemetry Datasets

The satellite telemetry data is organized into two annual directories, `annee1` and `annee2`.

Before integrating these datasets, we will examine their filenames, sizes, and structure to understand the available measurements.

This initial inspection will help determine how the datasets should be combined while preserving temporal consistency and avoiding unnecessary duplication.


In [1]:

from pathlib import Path
import pandas as pd

# Locate the project directory
project_root = Path.cwd().resolve()
data_dir = project_root / "data"

# Identify available CSV datasets
dataset_files = sorted(data_dir.glob("annee*/*.csv"))

# Create a dataset inventory
dataset_inventory = pd.DataFrame([
    {
        "Year Folder": file.parent.name,
        "Dataset": file.name,
        "Size (MB)": round(file.stat().st_size / (1024 ** 2), 2)
    }
    for file in dataset_files
])

print(f"Total CSV datasets found: {len(dataset_files)}")

display(dataset_inventory)


Total CSV datasets found: 12


,Year Folder,Dataset,Size (MB)
0,annee1,dmop.csv,4.63
1,annee1,evtf.csv,3.02
2,annee1,ftl.csv,0.83
3,annee1,ltdata.csv,0.06
4,annee1,power.csv,589.19
5,annee1,saaf.csv,44.08
6,annee2,dmop.csv,3.57
7,annee2,evtf.csv,2.86
8,annee2,ftl.csv,0.85
9,annee2,ltdata.csv,0.06



### 2.2.2 Dataset Structure Inspection

This section examines the structure of the available satellite telemetry datasets.

The inspection focuses on identifying column names, data types, and potential timestamp fields.

To minimize memory consumption, only a small sample of each CSV file will be loaded.

The results will guide subsequent decisions regarding dataset integration, temporal alignment, and feature preparation.


In [2]:

# Inspect dataset structures without loading entire files
dataset_structures = []

for file in dataset_files:

    # Read only the first five rows
    sample = pd.read_csv(file, nrows=5)

    dataset_structures.append({
        "Year Folder": file.parent.name,
        "Dataset": file.name,
        "Columns": sample.shape[1],
        "Column Names": list(sample.columns),
        "Data Types": sample.dtypes.astype(str).to_dict()
    })

# Display the structural information
for dataset in dataset_structures:
    print("=" * 70)
    print(f"Folder: {dataset['Year Folder']}")
    print(f"Dataset: {dataset['Dataset']}")
    print(f"Number of Columns: {dataset['Columns']}")
    print(f"Column Names: {dataset['Column Names']}")
    print(f"Data Types: {dataset['Data Types']}")
    print()


Folder: annee1
Dataset: dmop.csv
Number of Columns: 2
Column Names: ['ut_ms', 'subsystem']
Data Types: {'ut_ms': 'int64', 'subsystem': 'str'}

Folder: annee1
Dataset: evtf.csv
Number of Columns: 2
Column Names: ['ut_ms', 'description']
Data Types: {'ut_ms': 'int64', 'description': 'str'}

Folder: annee1
Dataset: ftl.csv
Number of Columns: 4
Column Names: ['utb_ms', 'ute_ms', 'type', 'flagcomms']
Data Types: {'utb_ms': 'int64', 'ute_ms': 'int64', 'type': 'str', 'flagcomms': 'bool'}

Folder: annee1
Dataset: ltdata.csv
Number of Columns: 7
Column Names: ['ut_ms', 'sunmars_km', 'earthmars_km', 'sunmarsearthangle_deg', 'solarconstantmars', 'eclipseduration_min', 'occultationduration_min']
Data Types: {'ut_ms': 'int64', 'sunmars_km': 'float64', 'earthmars_km': 'float64', 'sunmarsearthangle_deg': 'float64', 'solarconstantmars': 'float64', 'eclipseduration_min': 'float64', 'occultationduration_min': 'float64'}

Folder: annee1
Dataset: power.csv
Number of Columns: 34
Column Names: ['ut_ms', 'NP


### 2.2.3 Dataset Schema Compatibility

Before integrating satellite telemetry observations across the two annual datasets, schema compatibility must be verified.

This assessment compares column names, column order, and inferred data types between corresponding CSV files in `annee1` and `annee2`.

Consistent schemas support reliable integration, while structural differences may require additional preprocessing.

The comparison uses the previously inspected dataset samples without reloading the CSV files.


In [3]:

# Compare dataset schemas across the two annual folders

schema_records = {
    (item["Year Folder"], item["Dataset"]): item
    for item in dataset_structures
}

compatibility_results = []

for filename in sorted(dataset_inventory["Dataset"].unique()):

    year1 = schema_records.get(("annee1", filename))
    year2 = schema_records.get(("annee2", filename))

    if year1 is None or year2 is None:
        columns_match = False
        types_match = False
        status = "Missing dataset"
    else:
        columns_match = (
            year1["Column Names"] == year2["Column Names"]
        )

        types_match = (
            year1["Data Types"] == year2["Data Types"]
        )

        status = (
            "Compatible"
            if columns_match and types_match
            else "Review required"
        )

    compatibility_results.append({
        "Dataset": filename,
        "Columns Match": columns_match,
        "Data Types Match": types_match,
        "Status": status
    })

schema_compatibility = pd.DataFrame(compatibility_results)

display(schema_compatibility)


,Dataset,Columns Match,Data Types Match,Status
0,dmop.csv,True,True,Compatible
1,evtf.csv,True,True,Compatible
2,ftl.csv,True,True,Compatible
3,ltdata.csv,True,False,Review required
4,power.csv,True,True,Compatible
5,saaf.csv,True,True,Compatible


In [4]:

# Identify data type differences in ltdata.csv

ltdata_year1 = schema_records[("annee1", "ltdata.csv")]
ltdata_year2 = schema_records[("annee2", "ltdata.csv")]

dtype_comparison = pd.DataFrame({
    "Column": ltdata_year1["Column Names"],
    "Annee 1 Type": [
        ltdata_year1["Data Types"][col]
        for col in ltdata_year1["Column Names"]
    ],
    "Annee 2 Type": [
        ltdata_year2["Data Types"][col]
        for col in ltdata_year2["Column Names"]
    ]
})

dtype_comparison["Match"] = (
    dtype_comparison["Annee 1 Type"]
    == dtype_comparison["Annee 2 Type"]
)

display(dtype_comparison)


,Column,Annee 1 Type,Annee 2 Type,Match
0,ut_ms,int64,int64,True
1,sunmars_km,float64,float64,True
2,earthmars_km,float64,float64,True
3,sunmarsearthangle_deg,float64,float64,True
4,solarconstantmars,float64,float64,True
5,eclipseduration_min,float64,int64,False
6,occultationduration_min,float64,float64,True


In [5]:

# Validate eclipse duration values across both years

eclipse_validation = []

for year in ["annee1", "annee2"]:

    file_path = data_dir / year / "ltdata.csv"

    # Load only the required column
    eclipse_data = pd.read_csv(
        file_path,
        usecols=["eclipseduration_min"]
    )

    eclipse_numeric = pd.to_numeric(
        eclipse_data["eclipseduration_min"],
        errors="coerce"
    )

    eclipse_validation.append({
        "Year": year,
        "Rows": len(eclipse_data),
        "Inferred Type": str(
            eclipse_data["eclipseduration_min"].dtype
        ),
        "Missing or Non-numeric": int(
            eclipse_numeric.isna().sum()
        ),
        "Minimum Duration": eclipse_numeric.min(),
        "Maximum Duration": eclipse_numeric.max(),
        "Fractional Values": int(
            ((eclipse_numeric % 1 != 0)
             & eclipse_numeric.notna()).sum()
        )
    })

eclipse_validation_df = pd.DataFrame(eclipse_validation)

display(eclipse_validation_df)


,Year,Rows,Inferred Type,Missing or Non-numeric,Minimum Duration,Maximum Duration,Fractional Values
0,annee1,687,float64,0,0.0,55.450000,271
1,annee2,687,float64,0,0.0,48.583333,335



#### Schema Compatibility Validation — Findings

The initial schema comparison identified a data type difference in the `eclipseduration_min` variable between `annee1` and `annee2`.

Further examination of the complete eclipse-duration columns confirmed that:

- Both datasets contain 687 observations.
- Both columns are inferred as `float64` when fully loaded.
- Neither dataset contains missing or non-numeric eclipse-duration values.
- Fractional eclipse durations occur in both years.
- The initial discrepancy resulted from data type inference using only the first five observations.

**Preprocessing Decision:** Preserve `eclipseduration_min` as `float64` across both years to maintain numerical precision and ensure consistent data representation.

The investigated schema discrepancy is therefore resolved, and no modification of the original CSV files is required.



### 2.2.4 Timestamp Validation and Temporal Consistency

Satellite telemetry measurements are time-dependent. Accurate temporal information is therefore essential for integrating datasets and developing reliable AI models.

This section examines the timestamp coverage, ordering, and consistency of the available telemetry datasets.

The analysis will help establish whether observations can be aligned chronologically and identify potential issues that must be addressed before data integration.


In [6]:

# Inspect timestamp ranges across all datasets
# Read only timestamp columns to limit memory usage

timestamp_results = []

for file in dataset_files:

    header = pd.read_csv(file, nrows=0).columns.tolist()

    time_columns = [
        col for col in ["ut_ms", "utb_ms", "ute_ms"]
        if col in header
    ]

    for column in time_columns:

        first_timestamp = None
        last_timestamp = None
        min_timestamp = None
        max_timestamp = None
        total_rows = 0
        missing_count = 0
        non_monotonic_count = 0
        previous_timestamp = None

        for chunk in pd.read_csv(
            file,
            usecols=[column],
            chunksize=100_000
        ):
            timestamps = pd.to_numeric(
                chunk[column],
                errors="coerce"
            )

            total_rows += len(timestamps)
            missing_count += int(timestamps.isna().sum())

            valid = timestamps.dropna()

            if valid.empty:
                continue

            if first_timestamp is None:
                first_timestamp = valid.iloc[0]

            last_timestamp = valid.iloc[-1]

            chunk_min = valid.min()
            chunk_max = valid.max()

            min_timestamp = (
                chunk_min if min_timestamp is None
                else min(min_timestamp, chunk_min)
            )

            max_timestamp = (
                chunk_max if max_timestamp is None
                else max(max_timestamp, chunk_max)
            )

            # Check ordering within and between chunks
            non_monotonic_count += int(
                (valid.diff().dropna() < 0).sum()
            )

            if previous_timestamp is not None:
                non_monotonic_count += int(
                    valid.iloc[0] < previous_timestamp
                )

            previous_timestamp = valid.iloc[-1]

        timestamp_results.append({
            "Year": file.parent.name,
            "Dataset": file.name,
            "Timestamp Column": column,
            "Rows": total_rows,
            "Missing": missing_count,
            "First": first_timestamp,
            "Last": last_timestamp,
            "Minimum": min_timestamp,
            "Maximum": max_timestamp,
            "Out-of-Order": non_monotonic_count
        })

timestamp_validation_df = pd.DataFrame(timestamp_results)

display(timestamp_validation_df)


,Year,Dataset,Timestamp Column,Rows,Missing,First,Last,Minimum,Maximum,Out-of-Order
0,annee1,dmop.csv,ut_ms,187126,0,1219363211000,1278719847000,1219363211000,1278719847000,0
1,annee1,evtf.csv,ut_ms,76776,0,1219364253000,1278717961000,1219364253000,1278717961000,0
2,annee1,ftl.csv,utb_ms,19571,0,1219363213000,1278718460000,1219363213000,1278718460000,0
3,annee1,ftl.csv,ute_ms,19571,0,1219365494000,1278723155000,1219365494000,1278723155000,473
4,annee1,ltdata.csv,ut_ms,687,0,1219363200000,1278633600000,1219363200000,1278633600000,0
5,annee1,power.csv,ut_ms,1830121,0,1219363209542,1278719976556,1219363209542,1278719976556,0
6,annee1,saaf.csv,ut_ms,1261290,0,1219363213000,1278719990000,1219363213000,1278719990000,0
7,annee2,dmop.csv,ut_ms,143802,0,1278723363000,1336991940000,1278723363000,1336991940000,0
8,annee2,evtf.csv,ut_ms,75208,0,1278720876000,1338075725000,1278720876000,1338075725000,0
9,annee2,ftl.csv,utb_ms,20051,0,1278723155000,1338076762000,1278723155000,1338076762000,0



### 2.2.5 Flight Interval Validation

The `ftl.csv` datasets contain two timestamp variables: `utb_ms` and `ute_ms`.

Unlike individual telemetry observations, these variables appear to represent the beginning and ending timestamps of operational intervals.

The previous timestamp validation identified non-monotonic ending timestamps in both years.

This section investigates whether the intervals are internally consistent by checking their durations and identifying any intervals whose ending timestamps precede their beginning timestamps.

Non-monotonic ending timestamps will not automatically be treated as errors because operational intervals may overlap or have different durations.


In [7]:

# Validate start and end timestamps in flight intervals

interval_validation = []

for year in ["annee1", "annee2"]:

    ftl_path = data_dir / year / "ftl.csv"

    # Load only the two timestamp columns
    intervals = pd.read_csv(
        ftl_path,
        usecols=["utb_ms", "ute_ms"]
    )

    # Calculate interval durations in minutes
    duration_minutes = (
        intervals["ute_ms"] - intervals["utb_ms"]
    ) / 60_000

    interval_validation.append({
        "Year": year,
        "Total Intervals": len(intervals),
        "Negative Durations": int(
            (duration_minutes < 0).sum()
        ),
        "Zero Durations": int(
            (duration_minutes == 0).sum()
        ),
        "Minimum Duration (min)": duration_minutes.min(),
        "Maximum Duration (min)": duration_minutes.max(),
        "Median Duration (min)": duration_minutes.median()
    })

interval_validation_df = pd.DataFrame(interval_validation)

display(interval_validation_df)


,Year,Total Intervals,Negative Durations,Zero Durations,Minimum Duration (min),Maximum Duration (min),Median Duration (min)
0,annee1,19571,0,0,0.300000,1347.066667,30.000000
1,annee2,20051,0,0,0.766667,2008.300000,23.483333



#### Flight Interval Validation — Findings

The validation examined 19,571 operational intervals in `annee1` and 20,051 intervals in `annee2`.

The analysis confirmed that:

- All recorded intervals have positive durations.
- No negative or zero-duration intervals were identified.
- Median interval durations were 30.00 minutes in `annee1` and 23.48 minutes in `annee2`.
- Maximum durations reached approximately 1,347.07 minutes and 2,008.30 minutes, respectively.
- Non-monotonic ending timestamps do not indicate invalid start-to-end ordering.

**Preprocessing Decision:** Preserve the original start and end timestamps for each interval. Operational intervals will be represented using their duration and temporal boundaries rather than treating ending timestamps as an independently ordered time series.

The results support using these intervals in subsequent temporal integration, subject to further checks of their operational meaning and relationships.



### 2.2.6 Sampling Frequency and Temporal Alignment

Satellite telemetry datasets may be recorded at different sampling frequencies depending on the measurement type and operational requirements.

Before integrating these datasets, it is necessary to examine their observation intervals and identify differences in temporal resolution.

This analysis focuses on the time elapsed between consecutive observations and the consistency of sampling intervals.

The findings will guide the selection of an appropriate temporal alignment strategy while minimizing information loss and avoiding unnecessary interpolation.


In [8]:

# Analyze sampling intervals across timestamped datasets

sampling_results = []

for file in dataset_files:

    # Flight intervals are analyzed separately because
    # they represent start/end boundaries rather than
    # regularly sampled observations.
    if file.name == "ftl.csv":
        continue

    timestamps = pd.read_csv(
        file,
        usecols=["ut_ms"],
        dtype={"ut_ms": "int64"}
    )["ut_ms"]

    # Calculate time differences in seconds
    intervals_sec = timestamps.diff().dropna() / 1000

    positive_intervals = intervals_sec[intervals_sec > 0]

    sampling_results.append({
        "Year": file.parent.name,
        "Dataset": file.name,
        "Observations": len(timestamps),
        "Median Interval (sec)": positive_intervals.median(),
        "Mean Interval (sec)": positive_intervals.mean(),
        "Min Interval (sec)": positive_intervals.min(),
        "Max Interval (sec)": positive_intervals.max(),
        "Duplicate Timestamps": int(
            (intervals_sec == 0).sum()
        )
    })

sampling_frequency_df = pd.DataFrame(sampling_results)

display(sampling_frequency_df.round(2))


,Year,Dataset,Observations,Median Interval (sec),Mean Interval (sec),Min Interval (sec),Max Interval (sec),Duplicate Timestamps
0,annee1,dmop.csv,187126,13.0,317.20,1.00,1291141.0,0
1,annee1,evtf.csv,76776,345.0,773.77,1.00,9667.0,68
2,annee1,ltdata.csv,687,86400.0,86400.00,86400.00,86400.0,0
3,annee1,power.csv,1830121,32.0,32.43,3.97,26144.0,0
4,annee1,saaf.csv,1261290,60.0,47.06,1.00,60.0,0
5,annee2,dmop.csv,143802,14.0,405.20,1.00,3021771.0,0
6,annee2,evtf.csv,75208,349.0,790.24,1.00,9678.0,97
7,annee2,ltdata.csv,687,86400.0,86400.00,86400.00,86400.0,0
8,annee2,power.csv,1768248,32.0,33.57,0.00,83168.0,0
9,annee2,saaf.csv,1305759,40.0,45.46,1.00,9420.0,0



#### Sampling Frequency Analysis — Findings

The sampling frequency analysis revealed substantial differences in temporal resolution across the satellite telemetry datasets.

Key findings include:

- `power.csv` has a median sampling interval of 32 seconds in both years, although occasional large gaps occur.
- `ltdata.csv` contains regularly spaced daily observations with a sampling interval of 86,400 seconds.
- `saaf.csv` exhibits different median sampling intervals of 60 seconds in `annee1` and 40 seconds in `annee2`.
- `dmop.csv` and `evtf.csv` contain irregularly spaced operational records.
- `evtf.csv` contains repeated timestamps, potentially representing multiple events recorded at the same time.

**Preprocessing Implications:**

The datasets should not be integrated using row positions because their observation frequencies differ.

Instead, temporal alignment should preserve the meaning of each dataset, using timestamp-based matching, interval-aware representations, or carefully justified aggregation where appropriate.

Duplicate event timestamps should be investigated rather than automatically removed, and large telemetry gaps should be considered during model preparation.



### 2.2.7 Temporal Integration Strategy

The satellite telemetry datasets contain different sampling frequencies and represent distinct types of information.

Based on the preceding analyses, `power.csv` is proposed as the primary temporal reference for power-focused AI modeling.

The integration strategy will consider:

1. **Power telemetry:** Preserve the original measurement timestamps and identify irregular gaps.
2. **Environmental data:** Align daily measurements using their valid observation times without introducing future information.
3. **Numerical auxiliary telemetry:** Align `saaf.csv` measurements using appropriate timestamp matching and tolerance rules.
4. **Operational events:** Represent event information using time-aware features or event windows rather than arbitrary row-based joins.
5. **Flight intervals:** Preserve start and end boundaries and associate operational intervals with relevant telemetry observations.

The exact alignment methods will be determined after examining timestamp coverage, measurement semantics, and the requirements of the AI prediction task.

**Integration Principle:** Preserve temporal integrity, minimize information loss, and avoid data leakage during preprocessing.



## 2.3 AI Modeling Objective and Target Definition

### 2.3.1 Proposed Modeling Objective

The initial AI modeling objective is to predict future satellite power telemetry measurements using historical observations.

This approach will establish a data-driven foundation for the Dream-in-Orbit satellite simulation system.

The model will investigate whether historical power measurements and relevant contextual variables can be used to estimate future satellite electrical behavior.

### 2.3.2 Prediction Target

The initial prediction task will focus on one selected power telemetry variable.

The target variable will be chosen based on:

- Data availability and measurement quality.
- Temporal variability and predictability.
- Relationships with other telemetry variables.
- Relevance to satellite power system behavior, where supported by engineering documentation.

The prediction horizon will be established before constructing training examples.

### 2.3.3 Candidate Input Features

Potential model inputs include:

1. Historical power telemetry measurements.
2. Time-based features derived from observation timestamps.
3. Environmental and orbital context from `ltdata.csv`.
4. Auxiliary numerical telemetry from `saaf.csv`.
5. Operational information from event and interval datasets.

Feature inclusion will depend on data availability, temporal alignment, and whether the information would genuinely be available at prediction time.

### 2.3.4 Modeling Considerations

The modeling workflow will:

- Preserve chronological ordering.
- Avoid data leakage between training and testing periods.
- Evaluate predictions against simple baseline methods.
- Account for irregular sampling intervals and telemetry gaps.
- Distinguish observed measurements from simulated or predicted values.

### 2.3.5 Expected Outcome

The expected outcome is a validated forecasting model capable of estimating selected future satellite power measurements.

Its predictions may subsequently be incorporated into a broader satellite simulation framework.

This initial model will not, by itself, constitute a complete physical satellite simulator.



### 2.3.6 Selection of the Initial Prediction Target

Based on the exploratory analysis conducted in Notebook 01, `NPWD2721` is provisionally selected as the initial target variable for satellite power forecasting.

The selection is motivated by the following observations:

- The variable exhibits temporal fluctuations that provide an opportunity to investigate forecasting performance.
- It shows a moderate positive correlation with `NPWD2802` (r ≈ 0.439).
- Its historical behavior has already been examined through distribution, time-series, and monthly variability analyses.
- It provides a starting point for developing and evaluating the initial forecasting pipeline.

**Initial Target Variable:** `NPWD2721`

**Potential Supporting Variable:** `NPWD2802`

**Modeling Task:** Predict future values of `NPWD2721` using historical satellite telemetry observations.

This selection remains provisional until the engineering meaning of the telemetry variables is verified.

The next step is to establish the prediction horizon and determine how historical observations will be used to construct forecasting examples.



### 2.3.7 Definition of the Forecasting Horizon

The initial satellite power forecasting model will predict the value of `NPWD2721` five minutes into the future.

#### Forecasting Configuration

- **Target variable:** `NPWD2721`
- **Prediction horizon:** 5 minutes (300 seconds)
- **Primary dataset:** `power.csv`
- **Modeling approach:** Supervised time-series forecasting
- **Historical inputs:** Past satellite telemetry measurements available at the prediction time
- **Expected output:** Predicted value of `NPWD2721` approximately five minutes after the reference timestamp

#### Mathematical Formulation

The forecasting objective is expressed as:

\[
\{y}(t+300) = f(X_{<= t})
\]

Where:

- \({y}(t+300)\) represents the predicted value of `NPWD2721` at the future timestamp.
- \(t\) represents the current observation time in seconds.
- \(X_{<= t}\) represents historical telemetry information available up to time \(t\).
- \(f\) represents the forecasting model.

#### Handling Irregular Sampling

The exploratory analysis established that `power.csv` has a median sampling interval of approximately 32 seconds.

However, the dataset contains irregular sampling intervals and occasional large temporal gaps.

Therefore, the prediction horizon will be defined using actual timestamps rather than a fixed number of observation rows.

For each reference timestamp, a future target observation will be selected only when it falls within an explicitly defined tolerance around the five-minute forecasting horizon.

Observations without a suitable future target will be excluded from the supervised training examples rather than assigned an inaccurate target.

#### Data Leakage Prevention

Only information available at or before the prediction timestamp will be used as model inputs.

Future observations will be used exclusively to construct prediction targets and evaluate forecasting performance.

#### Expected Outcome

This configuration establishes a consistent forecasting objective for preparing the supervised learning dataset.

The next stage will examine timestamp spacing and determine a suitable tolerance for matching future observations.



### 2.3.8 Future Target Matching and Timestamp Tolerance

The forecasting objective requires predicting `NPWD2721` five minutes (300 seconds) into the future.

However, satellite power telemetry is not recorded at perfectly uniform intervals. Therefore, an observation may not exist at exactly 300 seconds after a given reference timestamp.

This section evaluates the temporal difference between the desired prediction time and the nearest available future measurement.

The analysis will:

1. Identify the closest observation to the five-minute prediction horizon.
2. Calculate the absolute timestamp matching error.
3. Examine the distribution of matching errors.
4. Determine an appropriate tolerance for constructing supervised learning targets.

The tolerance will be selected based on observed timestamp behavior rather than an arbitrary assumption.

Only measurements recorded after the reference timestamp will be considered as future targets.


In [9]:

import numpy as np

# Forecasting horizon in milliseconds
forecast_horizon_ms = 300_000

matching_results = []

for year in ["annee1", "annee2"]:

    power_path = data_dir / year / "power.csv"

    # Read timestamps only
    timestamps = pd.read_csv(
        power_path,
        usecols=["ut_ms"],
        dtype={"ut_ms": "int64"}
    )["ut_ms"].to_numpy()

    # Ensure chronological ordering
    timestamps = np.sort(timestamps)

    # Desired future timestamps
    target_times = timestamps + forecast_horizon_ms

    # Find positions where future observations would occur
    positions = np.searchsorted(
        timestamps,
        target_times,
        side="left"
    )

    # Compare observations immediately before and after
    # each desired future timestamp
    right_idx = np.clip(
        positions, 0, len(timestamps) - 1
    )

    left_idx = np.clip(
        positions - 1, 0, len(timestamps) - 1
    )

    right_error = np.abs(
        timestamps[right_idx] - target_times
    )

    left_error = np.abs(
        timestamps[left_idx] - target_times
    )

    # Select the observation closest to the desired time
    nearest_idx = np.where(
        left_error <= right_error,
        left_idx,
        right_idx
    )

    actual_future_times = timestamps[nearest_idx]

    # Ensure selected targets occur after reference times
    valid_future = actual_future_times > timestamps

    matching_error_sec = (
        np.abs(actual_future_times - target_times) / 1000
    )

    valid_errors = matching_error_sec[valid_future]

    matching_results.append({
        "Year": year,
        "Reference Observations": len(timestamps),
        "Valid Future Matches": int(valid_future.sum()),
        "Median Error (sec)": np.median(valid_errors),
        "95th Percentile Error (sec)": np.percentile(
            valid_errors, 95
        ),
        "99th Percentile Error (sec)": np.percentile(
            valid_errors, 99
        ),
        "Maximum Error (sec)": np.max(valid_errors)
    })

target_matching_df = pd.DataFrame(matching_results)

display(target_matching_df.round(2))


,Year,Reference Observations,Valid Future Matches,Median Error (sec),95th Percentile Error (sec),99th Percentile Error (sec),Maximum Error (sec)
0,annee1,1830121,1830100,12.0,12.01,20.00,268.0
1,annee2,1768248,1768179,12.0,12.01,12.04,276.0



### 2.3.9 Selection of the Future Target Matching Tolerance

The previous analysis showed that most future satellite power measurements occur close to the desired five-minute forecasting horizon.

A candidate matching tolerance of ±20 seconds is proposed for evaluation.

This corresponds to accepting future target observations recorded between 280 and 320 seconds after the reference timestamp.

The analysis will compare several tolerance values and determine how many observations can be retained for supervised learning.

The final tolerance will balance temporal accuracy against the availability of training examples.


In [10]:

# Evaluate candidate target-matching tolerances

candidate_tolerances = [5, 10, 15, 20, 30]
tolerance_results = []

for year in ["annee1", "annee2"]:

    power_path = data_dir / year / "power.csv"

    timestamps = pd.read_csv(
        power_path,
        usecols=["ut_ms"],
        dtype={"ut_ms": "int64"}
    )["ut_ms"].to_numpy()

    timestamps = np.sort(timestamps)

    target_times = timestamps + forecast_horizon_ms

    positions = np.searchsorted(
        timestamps, target_times, side="left"
    )

    right_idx = np.clip(positions, 0, len(timestamps) - 1)
    left_idx = np.clip(positions - 1, 0, len(timestamps) - 1)

    right_error = np.abs(timestamps[right_idx] - target_times)
    left_error = np.abs(timestamps[left_idx] - target_times)

    nearest_idx = np.where(
        left_error <= right_error,
        left_idx,
        right_idx
    )

    actual_times = timestamps[nearest_idx]
    errors_sec = np.abs(actual_times - target_times) / 1000

    valid_future = actual_times > timestamps

    for tolerance in candidate_tolerances:

        accepted = valid_future & (errors_sec <= tolerance)

        tolerance_results.append({
            "Year": year,
            "Tolerance (sec)": tolerance,
            "Accepted Matches": int(accepted.sum()),
            "Acceptance Rate (%)": round(
                100 * accepted.mean(), 2
            )
        })

tolerance_comparison_df = pd.DataFrame(tolerance_results)

display(tolerance_comparison_df)


,Year,Tolerance (sec),Accepted Matches,Acceptance Rate (%)
0,annee1,5,2926,0.16
1,annee1,10,3054,0.17
2,annee1,15,1808760,98.83
3,annee1,20,1826339,99.79
4,annee1,30,1829639,99.97
5,annee2,5,3789,0.21
6,annee2,10,3954,0.22
7,annee2,15,1752873,99.13
8,annee2,20,1752931,99.13
9,annee2,30,1753254,99.15



### 2.3.10 Final Forecasting Horizon and Matching Tolerance

The future target-matching analysis evaluated five candidate tolerances: ±5, ±10, ±15, ±20, and ±30 seconds.

The results showed that a ±20-second tolerance provides a suitable balance between temporal accuracy and observation retention.

#### Final Forecasting Configuration

| Parameter | Selected Value |
|---|---|
| Target variable | `NPWD2721` |
| Forecasting horizon | 300 seconds (5 minutes) |
| Matching tolerance | ±20 seconds |
| Accepted target window | 280–320 seconds |
| Primary dataset | `power.csv` |
| Forecasting approach | Supervised time-series learning |

#### Supporting Results

- **Année 1:** 1,826,339 accepted timestamp matches (99.79%).
- **Année 2:** 1,752,931 accepted timestamp matches (99.13%).
- A tolerance of ±15 seconds already retains more than 98% of observations in both years.
- Increasing the tolerance beyond ±20 seconds produces limited additional improvement.

#### Final Decision

A ±20-second tolerance will be adopted for constructing the initial supervised forecasting targets.

The nearest available future measurement will be selected only when its timestamp falls within the accepted prediction window.

The subsequent preprocessing stages will verify target availability, temporal boundaries, and data quality before generating model-ready training examples.



## 2.4 Construction of the Supervised Forecasting Dataset

### 2.4.1 Understanding Supervised Forecasting Data

Supervised machine learning requires input features and corresponding target values.

For satellite power forecasting, historical telemetry measurements will serve as model inputs, while future measurements of the selected power variable will serve as prediction targets.

#### Forecasting Structure

For each reference timestamp \(t\):

- **Input features (X):** Satellite telemetry measurements available at or before time \(t\).
- **Prediction target (y):** The observed value of `NPWD2721` approximately 300 seconds after time \(t\).
- **Matching tolerance:** ±20 seconds around the desired prediction time.

The forecasting relationship is expressed as:

\[
\{y}(t+300) = f(X_{<=t})
\]

#### Example

| Reference Time | Input NPWD2721 | Target Time | Future NPWD2721 |
|---|---|---|---|
| 12:00:00 | 15.2 | 12:05:08 | 16.1 |
| 12:00:32 | 15.5 | 12:05:20 | 15.9 |
| 12:01:04 | 15.3 | 12:06:00 | 16.4 |

*Note: These values and timestamps are illustrative and do not represent actual satellite measurements.*

#### Dataset Construction Requirements

The preprocessing workflow will:

1. Load the required power telemetry columns.
2. Preserve chronological ordering.
3. Identify future observations within the accepted forecasting window.
4. Associate each reference observation with its corresponding future target.
5. Exclude observations without valid future targets.
6. Preserve the original timestamps for temporal validation.
7. Keep Année 1 and Année 2 separate during initial target construction to prevent cross-year matching.

The resulting dataset will establish the foundation for feature engineering, chronological data splitting, and AI model training.



### 2.4.2 Loading the Required Power Telemetry Columns

The initial forecasting dataset will be constructed using selected satellite power telemetry variables from `power.csv`.

To reduce memory consumption, only the columns required for the initial modeling workflow will be loaded.

#### Selected Columns

| Column | Purpose |
|---|---|
| `ut_ms` | Observation timestamp in Unix milliseconds |
| `NPWD2721` | Primary forecasting target |
| `NPWD2802` | Candidate supporting input feature |

The datasets from Année 1 and Année 2 will initially be loaded separately.

Each dataset will be sorted chronologically, and timestamp uniqueness and target availability will be verified before future target matching.

The original telemetry values will be preserved without applying scaling or other transformations at this stage.


In [11]:

# Section 2.4.2 — Load selected power telemetry columns

selected_power_columns = [
    "ut_ms",
    "NPWD2721",
    "NPWD2802"
]

power_data = {}
loading_summary = []

for year in ["annee1", "annee2"]:

    power_path = data_dir / year / "power.csv"

    # Load only the required telemetry columns
    df = pd.read_csv(
        power_path,
        usecols=selected_power_columns,
        dtype={
            "ut_ms": "int64",
            "NPWD2721": "float64",
            "NPWD2802": "float64"
        }
    )

    # Preserve chronological ordering
    df = df.sort_values("ut_ms").reset_index(drop=True)

    # Store each year separately
    power_data[year] = df

    # Summarize loading and initial data quality
    loading_summary.append({
        "Year": year,
        "Observations": len(df),
        "Columns": df.shape[1],
        "Duplicate Timestamps": int(
            df["ut_ms"].duplicated().sum()
        ),
        "Missing NPWD2721": int(
            df["NPWD2721"].isna().sum()
        ),
        "Missing NPWD2802": int(
            df["NPWD2802"].isna().sum()
        ),
        "Chronologically Sorted": bool(
            df["ut_ms"].is_monotonic_increasing
        )
    })

power_loading_summary = pd.DataFrame(loading_summary)

display(power_loading_summary)


,Year,Observations,Columns,Duplicate Timestamps,Missing NPWD2721,Missing NPWD2802,Chronologically Sorted
0,annee1,1830121,3,0,0,0,True
1,annee2,1768248,3,0,0,0,True



### 2.4.3 Validation of Forecasting Inputs

The initial data-loading assessment confirmed that both annual power datasets contain unique timestamps, no missing values in the selected telemetry variables, and chronologically ordered observations.

Before constructing supervised forecasting examples, an additional numerical validation will be performed.

This validation will:

1. Check for positive or negative infinite values.
2. Confirm that all selected power measurements are finite.
3. Verify that timestamps are strictly increasing.
4. Establish whether the loaded data can proceed to future target construction.

The validation will reuse the datasets already loaded into memory, avoiding unnecessary repeated file reading.

No telemetry measurements will be modified during this step.


In [12]:

# Section 2.4.3 — Validate forecasting input measurements

validation_results = []

for year, df in power_data.items():

    power_values = df[["NPWD2721", "NPWD2802"]]

    # Check for non-finite numerical values
    finite_mask = np.isfinite(power_values.to_numpy())

    # Verify strict chronological ordering
    timestamp_differences = np.diff(
        df["ut_ms"].to_numpy()
    )

    validation_results.append({
        "Year": year,
        "Observations": len(df),
        "Non-finite NPWD2721": int(
            (~finite_mask[:, 0]).sum()
        ),
        "Non-finite NPWD2802": int(
            (~finite_mask[:, 1]).sum()
        ),
        "Strictly Increasing Timestamps": bool(
            np.all(timestamp_differences > 0)
        ),
        "All Power Values Finite": bool(
            finite_mask.all()
        )
    })

forecast_input_validation = pd.DataFrame(
    validation_results
)

display(forecast_input_validation)


,Year,Observations,Non-finite NPWD2721,Non-finite NPWD2802,Strictly Increasing Timestamps,All Power Values Finite
0,annee1,1830121,0,0,True,True
1,annee2,1768248,0,0,True,True



### 2.4.4 Construction of Five-Minute Forecasting Targets

The validated power telemetry datasets will now be transformed into supervised forecasting examples.

For each reference observation, the algorithm will identify the measurement closest to the desired five-minute prediction timestamp.

#### Target Construction Rules

1. Use `NPWD2721` as the prediction target.
2. Define the desired future timestamp as the reference timestamp plus 300 seconds.
3. Find the nearest available observation to the desired future timestamp.
4. Accept the observation only if its timestamp is between 280 and 320 seconds after the reference timestamp.
5. Preserve the actual future timestamp and prediction interval.
6. Construct forecasting targets separately for Année 1 and Année 2.
7. Exclude observations without an acceptable future measurement.

The resulting datasets will contain the current telemetry measurements and their corresponding future `NPWD2721` values.

These input-target pairs will subsequently be used for feature engineering and supervised model training.


In [13]:

# Section 2.4.4 — Construct five-minute forecasting targets

matching_tolerance_ms = 20_000

forecast_data = {}
forecast_summary = []

for year, df in power_data.items():

    timestamps = df["ut_ms"].to_numpy()

    # Desired prediction timestamps
    desired_times = timestamps + forecast_horizon_ms

    # Locate the nearest available future measurements
    positions = np.searchsorted(
        timestamps, desired_times, side="left"
    )

    right_idx = np.clip(positions, 0, len(df) - 1)
    left_idx = np.clip(positions - 1, 0, len(df) - 1)

    right_error = np.abs(
        timestamps[right_idx] - desired_times
    )

    left_error = np.abs(
        timestamps[left_idx] - desired_times
    )

    nearest_idx = np.where(
        left_error <= right_error,
        left_idx,
        right_idx
    )

    matched_times = timestamps[nearest_idx]

    # Apply the future-time and tolerance requirements
    valid_matches = (
        (matched_times > timestamps)
        & (np.abs(matched_times - desired_times)
           <= matching_tolerance_ms)
    )

    # Preserve current observations with valid future targets
    forecast_df = df.loc[valid_matches].copy()

    matched_indices = nearest_idx[valid_matches]

    forecast_df["target_ut_ms"] = (
        timestamps[matched_indices]
    )

    forecast_df["target_NPWD2721"] = (
        df["NPWD2721"].to_numpy()[matched_indices]
    )

    forecast_df["forecast_interval_sec"] = (
        forecast_df["target_ut_ms"].to_numpy()
        - forecast_df["ut_ms"].to_numpy()
    ) / 1000

    forecast_df = forecast_df.reset_index(drop=True)

    # Store each year's forecasting dataset
    forecast_data[year] = forecast_df

    forecast_summary.append({
        "Year": year,
        "Original Observations": len(df),
        "Forecasting Examples": len(forecast_df),
        "Excluded Observations": (
            len(df) - len(forecast_df)
        ),
        "Retention Rate (%)": round(
            100 * len(forecast_df) / len(df), 2
        ),
        "Min Interval (sec)": (
            forecast_df["forecast_interval_sec"].min()
        ),
        "Max Interval (sec)": (
            forecast_df["forecast_interval_sec"].max()
        )
    })

forecast_construction_summary = pd.DataFrame(
    forecast_summary
)

display(forecast_construction_summary.round(2))


,Year,Original Observations,Forecasting Examples,Excluded Observations,Retention Rate (%),Min Interval (sec),Max Interval (sec)
0,annee1,1830121,1826339,3782,99.79,284.98,320.0
1,annee2,1768248,1752931,15317,99.13,284.98,320.0



### 2.4.5 Validation of Temporal Continuity

The previous section generated 3,579,270 candidate forecasting examples across the two annual datasets.

Although all accepted future targets fall within the specified forecasting window, timestamp matching alone does not guarantee continuous telemetry coverage.

This section investigates whether large measurement gaps occur between reference observations and their corresponding future targets.

The analysis will:

1. Calculate consecutive timestamp differences.
2. Identify forecasting examples containing gaps exceeding selected diagnostic thresholds.
3. Compare the frequency of temporal interruptions between the two years.
4. Determine whether additional continuity filtering is necessary.

Gap thresholds will initially be used for diagnostic purposes. No forecasting examples will be removed until the results have been evaluated.


In [14]:

# Section 2.4.5 — Diagnose temporal gaps in forecasting examples

gap_thresholds_sec = [60, 120, 300]
continuity_results = []

for year, df in power_data.items():

    timestamps = df["ut_ms"].to_numpy()
    forecast_df = forecast_data[year]

    # Consecutive observation intervals
    gaps_ms = np.diff(timestamps)

    # Locate reference and target positions
    reference_idx = np.searchsorted(
        timestamps,
        forecast_df["ut_ms"].to_numpy()
    )

    target_idx = np.searchsorted(
        timestamps,
        forecast_df["target_ut_ms"].to_numpy()
    )

    for threshold_sec in gap_thresholds_sec:

        # Flag consecutive gaps exceeding the threshold
        large_gaps = gaps_ms > threshold_sec * 1000

        # Cumulative counts allow efficient interval checks
        cumulative_gaps = np.concatenate((
            [0],
            np.cumsum(large_gaps, dtype=np.int64)
        ))

        # Count large gaps between each reference and target
        gaps_per_example = (
            cumulative_gaps[target_idx]
            - cumulative_gaps[reference_idx]
        )

        affected = gaps_per_example > 0

        continuity_results.append({
            "Year": year,
            "Gap Threshold (sec)": threshold_sec,
            "Examples Examined": len(forecast_df),
            "Examples Crossing Gaps": int(affected.sum()),
            "Affected (%)": round(
                100 * affected.mean(), 2
            )
        })

temporal_continuity_df = pd.DataFrame(
    continuity_results
)

display(temporal_continuity_df)


,Year,Gap Threshold (sec),Examples Examined,Examples Crossing Gaps,Affected (%)
0,annee1,60,1826339,19967,1.09
1,annee1,120,1826339,157,0.01
2,annee1,300,1826339,0,0.00
3,annee2,60,1752931,2840,0.16
4,annee2,120,1752931,52,0.00
5,annee2,300,1752931,0,0.00



### 2.4.6 Filtering Forecasting Examples with Large Temporal Gaps

The temporal continuity analysis identified a small number of forecasting examples containing substantial gaps between consecutive telemetry measurements.

A maximum permitted consecutive measurement gap of 120 seconds has been selected for the initial forecasting dataset.

#### Filtering Criteria

A forecasting example will be retained only if:

1. Its future target satisfies the previously established 280–320-second prediction window.
2. No consecutive telemetry interval between the reference observation and its target exceeds 120 seconds.
3. The reference and target observations belong to the same annual dataset.

#### Rationale

The 120-second threshold removes examples with substantial telemetry interruptions while preserving nearly all candidate forecasting observations.

The filtering process will preserve the original telemetry values and will not interpolate or artificially generate measurements.

This criterion applies to the interval between each reference observation and its future target. Additional continuity requirements may be necessary when historical input sequences are constructed.


In [15]:

# Section 2.4.6 — Filter forecasting examples with large gaps

max_allowed_gap_ms = 120_000

forecast_data_filtered = {}
filtering_summary = []

for year, df in power_data.items():

    timestamps = df["ut_ms"].to_numpy()
    forecast_df = forecast_data[year]

    # Identify consecutive telemetry gaps above 120 seconds
    large_gaps = np.diff(timestamps) > max_allowed_gap_ms

    # Cumulative count of large gaps
    cumulative_gaps = np.concatenate([
        [0],
        np.cumsum(large_gaps, dtype=np.int64)
    ])

    # Locate reference and future target positions
    reference_idx = np.searchsorted(
        timestamps,
        forecast_df["ut_ms"].to_numpy()
    )

    target_idx = np.searchsorted(
        timestamps,
        forecast_df["target_ut_ms"].to_numpy()
    )

    # Count large gaps within each forecasting interval
    gaps_per_example = (
        cumulative_gaps[target_idx]
        - cumulative_gaps[reference_idx]
    )

    # Keep examples without gaps exceeding 120 seconds
    valid_continuity = gaps_per_example == 0

    filtered_df = (
        forecast_df.loc[valid_continuity]
        .reset_index(drop=True)
    )

    forecast_data_filtered[year] = filtered_df

    filtering_summary.append({
        "Year": year,
        "Before Filtering": len(forecast_df),
        "Removed": int((~valid_continuity).sum()),
        "After Filtering": len(filtered_df),
        "Retention (%)": round(
            100 * len(filtered_df) / len(forecast_df), 4
        )
    })

continuity_filter_summary = pd.DataFrame(
    filtering_summary
)

display(continuity_filter_summary)


,Year,Before Filtering,Removed,After Filtering,Retention (%)
0,annee1,1826339,157,1826182,99.9914
1,annee2,1752931,52,1752879,99.9970



### 2.4.7 Final Forecasting Dataset Validation

Following temporal continuity filtering, the forecasting datasets are validated to ensure their suitability for subsequent feature engineering and machine learning.

The validation verifies:

1. All retained examples have valid reference and future target timestamps.
2. Forecasting intervals remain within the accepted 280–320-second window.
3. No missing or non-finite values exist in the selected power variables or prediction target.
4. Each future target timestamp occurs after its corresponding reference timestamp.

The purpose is to confirm data integrity before constructing historical input features and preparing training and evaluation datasets.


In [16]:

# Section 2.4.7 — Final forecasting dataset validation

final_validation_results = []

for year, df in forecast_data_filtered.items():

    # Check the prediction interval
    actual_interval = (
        df["target_ut_ms"] - df["ut_ms"]
    ) / 1000

    # Check all numeric columns for missing or non-finite values
    numeric_columns = df.select_dtypes(
        include="number"
    )

    non_finite_rows = (
        ~np.isfinite(numeric_columns.to_numpy())
    ).any(axis=1)

    final_validation_results.append({
        "Year": year,
        "Total Examples": len(df),
        "Invalid Time Order": int(
            (actual_interval <= 0).sum()
        ),
        "Outside 280–320 sec": int(
            (~actual_interval.between(280, 320)).sum()
        ),
        "Rows with Non-finite Values": int(
            non_finite_rows.sum()
        ),
        "Duplicate Reference Times": int(
            df["ut_ms"].duplicated().sum()
        )
    })

final_validation_df = pd.DataFrame(
    final_validation_results
)

display(final_validation_df)


,Year,Total Examples,Invalid Time Order,Outside 280–320 sec,Rows with Non-finite Values,Duplicate Reference Times
0,annee1,1826182,0,0,0,0
1,annee2,1752879,0,0,0,0



### 2.4.8 Forecasting Dataset Preparation Summary

The forecasting dataset preparation stage has been completed successfully for both annual satellite datasets.

**Prediction configuration:**
- Target variable: `NPWD2721`
- Supporting power variable: `NPWD2802`
- Prediction horizon: 5 minutes (300 seconds)
- Accepted prediction interval: 280–320 seconds
- Maximum permitted temporal gap: 120 seconds

**Final dataset:**

| Dataset | Valid Forecasting Examples |
|---|---:|
| Année 1 | 1,826,182 |
| Année 2 | 1,752,879 |
| **Total** | **3,579,061** |

**Key findings:**

1. Both annual power datasets contain valid, chronologically ordered timestamps.
2. The selected power measurements contain no missing or non-finite values.
3. Future prediction targets were matched using timestamp-based alignment to accommodate irregular sampling intervals.
4. Temporal continuity filtering removed 209 examples across both years.
5. All retained examples satisfy the accepted forecasting interval.
6. Final validation confirmed no invalid timestamp order, non-finite values, or duplicate reference timestamps.

**Conclusion:**

The resulting datasets provide a validated foundation for short-term satellite power forecasting. The next stage will construct historical features using only information available at each prediction time, preventing future information leakage during model development.



## 2.5 Historical Feature Engineering

### 2.5.1 Historical Feature Engineering Strategy

Feature engineering transforms historical satellite power measurements into informative inputs for machine learning models.

The objective is to predict the future value of `NPWD2721` approximately five minutes ahead using only measurements available at the prediction time.

#### 1. Forecasting Objective

The prediction target is:

$$
y(t) = NPWD2721(t + 300)
$$

where:

- $t$ represents the current observation time.
- $300$ represents the five-minute forecasting horizon in seconds.
- $y(t)$ represents the future satellite power measurement.

Because the satellite data have irregular sampling intervals, future targets are matched using actual timestamps rather than a fixed number of rows.

The accepted forecasting interval is 280–320 seconds.

#### 2. Lag Features

Lag features represent historical measurements of a variable.

For a power variable $P$, a time-based lag is defined as:

$$
P_{\mathrm{lag},\Delta}(t) = P(t-\Delta)
$$

where $\Delta$ is the historical time offset.

We will consider historical measurements from:

- `NPWD2721`: the target power variable.
- `NPWD2802`: a supporting power variable identified during exploratory analysis.

Lagged measurements can help the model recognize recent operating conditions and changes in power behavior.

#### 3. Rolling Statistical Features

Rolling features summarize historical measurements over a specified time window.

**Rolling mean:**

$$
\mu_W(t) = \frac{1}{N_W}\sum_{i \in \mathcal{H}_W(t)} P(t_i)
$$

**Rolling standard deviation:**

$$
\sigma_W(t) =
\sqrt{
\frac{1}{N_W}
\sum_{i \in \mathcal{H}_W(t)}
\left(P(t_i)-\mu_W(t)\right)^2
}
$$

where:

- $W$ represents the historical window duration.
- $\mathcal{H}_W(t)$ contains observations within the window ending at time $t$.
- $N_W$ is the number of observations in that window.
- $\mu_W(t)$ measures recent average power.
- $\sigma_W(t)$ measures recent power variability.

These features may help the model identify stable and changing operating conditions.

#### 4. Preventing Data Leakage

All input features must be calculated using measurements available at or before the prediction time.

For every forecasting example:

$$
t_{\mathrm{feature}} \leq t_{\mathrm{prediction}} < t_{\mathrm{target}}
$$

Future measurements must never be used to construct historical input features.

#### 5. Feature Engineering Principles

The following principles will guide implementation:

1. Use actual timestamps to account for irregular sampling.
2. Reuse the validated forecasting datasets from Section 2.4.
3. Construct features independently for each annual dataset.
4. Avoid unnecessary duplication of existing preprocessing.
5. Prevent future information leakage.
6. Validate engineered features before model training.

The next step is to define appropriate historical lag intervals and rolling window durations.



### 2.5.2 Selection of Historical Lag Intervals and Rolling Windows

Historical features are selected to capture short-term satellite power behavior before the five-minute prediction horizon.

#### Selected Input Variables

- `NPWD2721`: primary power variable and forecasting target.
- `NPWD2802`: supporting power variable.

#### Historical Lag Intervals

Three historical lag intervals are proposed:

- 60 seconds (1 minute)
- 120 seconds (2 minutes)
- 300 seconds (5 minutes)

For each variable, lag features represent previously observed power measurements at the selected historical offsets.

Current measurements at the prediction timestamp will also be retained as input features.

#### Rolling Statistical Windows

Two historical window durations are proposed:

- 120 seconds (2 minutes)
- 300 seconds (5 minutes)

For each window, the following statistics will be calculated:

1. Rolling mean: average power within the historical window.
2. Rolling standard deviation: variability of power measurements within the historical window.

#### Timestamp-Aware Implementation

Because the satellite power measurements are irregularly sampled, lag features will be matched using timestamps rather than fixed row offsets.

Historical matches must not use measurements occurring after the prediction timestamp.

Rolling statistics will include only observations within their respective historical windows.

#### Temporal Continuity

Historical feature construction will account for temporal gaps. Lag matches will require an appropriate timestamp tolerance, and rolling windows will be checked for adequate historical coverage.

This prevents the model from interpreting widely separated measurements as continuous satellite behavior.

#### Initial Feature Configuration

| Feature Category | Configuration |
|---|---|
| Input variables | NPWD2721, NPWD2802 |
| Current measurements | Both variables |
| Historical lags | 60, 120, 300 seconds |
| Rolling windows | 120, 300 seconds |
| Rolling statistics | Mean, standard deviation |
| Prediction horizon | 300 seconds |
| Target matching tolerance | 20 seconds |

The selected configuration will serve as the initial feature engineering strategy. Its effectiveness will be assessed during model evaluation.



### 2.5.3 Timestamp-Aware Historical Lag Feature Construction

Historical lag features are constructed using timestamp-based matching to account for irregular satellite power sampling intervals.

For each prediction timestamp, historical measurements of `NPWD2721` and `NPWD2802` are retrieved at approximately 60, 120, and 300 seconds before the prediction time.

A backward-looking timestamp match is used to ensure that no observation occurring after the requested historical timestamp is selected.

A maximum matching tolerance of 40 seconds is applied. Historical measurements outside this tolerance are treated as unavailable.

The original forecasting datasets remain unchanged. Lag features are stored in separate feature-engineered datasets for subsequent validation and rolling feature construction.


In [17]:

# Section 2.5.3 — Timestamp-aware historical lag features

lag_intervals = [60, 120, 300]
lag_tolerance_ms = 40_000

lag_feature_data = {}

for year, forecast_df in forecast_data_filtered.items():

    # Preserve the validated forecasting dataset
    features_df = forecast_df.copy()

    # Retrieve the corresponding historical power measurements
    history_df = power_data[year][
        ["ut_ms", "NPWD2721", "NPWD2802"]
    ].sort_values("ut_ms").copy()

    # Construct lag features using actual timestamps
    for lag_sec in lag_intervals:

        lookup_df = features_df[
            ["ut_ms"]
        ].copy()

        lookup_df["historical_timestamp"] = (
            lookup_df["ut_ms"] - lag_sec * 1000
        )

        historical_values = history_df.rename(
            columns={
                "ut_ms": "historical_timestamp",
                "NPWD2721": f"NPWD2721_lag_{lag_sec}s",
                "NPWD2802": f"NPWD2802_lag_{lag_sec}s"
            }
        )

        matched = pd.merge_asof(
            lookup_df.sort_values("historical_timestamp"),
            historical_values.sort_values("historical_timestamp"),
            on="historical_timestamp",
            direction="backward",
            tolerance=lag_tolerance_ms
        )

        matched = matched.sort_index()

        features_df[
            f"NPWD2721_lag_{lag_sec}s"
        ] = matched[
            f"NPWD2721_lag_{lag_sec}s"
        ].to_numpy()

        features_df[
            f"NPWD2802_lag_{lag_sec}s"
        ] = matched[
            f"NPWD2802_lag_{lag_sec}s"
        ].to_numpy()

    lag_feature_data[year] = features_df

    print(
        f"{year}: {len(features_df):,} examples, "
        f"{len(features_df.columns)} columns"
    )


annee1: 1,826,182 examples, 12 columns
annee2: 1,752,879 examples, 12 columns



### 2.5.4 Historical Lag Feature Validation

The historical lag features are evaluated to determine whether sufficient historical power measurements are available for each forecasting example.

The validation examines:

1. Missing values in each lag feature.
2. The percentage of available historical measurements.
3. The number of forecasting examples with complete lag information.

Missing lag values may occur when historical observations are unavailable within the specified matching tolerance.

These values will be assessed before deciding how to prepare the final machine learning dataset.


In [18]:

# Section 2.5.4 — Historical lag feature validation

lag_columns = [
    f"{variable}_lag_{lag_sec}s"
    for variable in ["NPWD2721", "NPWD2802"]
    for lag_sec in lag_intervals
]

lag_validation_results = []
lag_completeness_results = []

for year, df in lag_feature_data.items():

    # Check availability of each historical lag feature
    for column in lag_columns:

        missing = int(df[column].isna().sum())

        lag_validation_results.append({
            "Year": year,
            "Lag Feature": column,
            "Missing Values": missing,
            "Availability (%)": round(
                100 * (1 - missing / len(df)), 2
            )
        })

    # Check whether each example has all six lag features
    complete_mask = df[lag_columns].notna().all(axis=1)

    lag_completeness_results.append({
        "Year": year,
        "Total Examples": len(df),
        "Complete Lag Examples": int(complete_mask.sum()),
        "Incomplete Lag Examples": int((~complete_mask).sum()),
        "Completeness (%)": round(
            100 * complete_mask.mean(), 2
        )
    })

lag_validation_df = pd.DataFrame(lag_validation_results)
lag_completeness_df = pd.DataFrame(lag_completeness_results)

print("Historical Lag Feature Availability")
display(lag_validation_df)

print("Overall Historical Lag Completeness")
display(lag_completeness_df)


Historical Lag Feature Availability


,Year,Lag Feature,Missing Values,Availability (%)
0,annee1,NPWD2721_lag_60s,140,99.99
1,annee1,NPWD2721_lag_120s,278,99.98
2,annee1,NPWD2721_lag_300s,817,99.96
3,annee1,NPWD2802_lag_60s,140,99.99
4,annee1,NPWD2802_lag_120s,278,99.98
5,annee1,NPWD2802_lag_300s,817,99.96
6,annee2,NPWD2721_lag_60s,83,100.00
7,annee2,NPWD2721_lag_120s,184,99.99
8,annee2,NPWD2721_lag_300s,720,99.96
9,annee2,NPWD2802_lag_60s,83,100.00


Overall Historical Lag Completeness


,Year,Total Examples,Complete Lag Examples,Incomplete Lag Examples,Completeness (%)
0,annee1,1826182,1825206,976,99.95
1,annee2,1752879,1752092,787,99.96



### 2.5.5 Historical Lag Feature Interpretation

The timestamp-aware historical lag construction successfully generated six lag features using measurements from `NPWD2721` and `NPWD2802`.

The historical offsets considered were 60, 120, and 300 seconds.

#### Validation Results

| Metric | Année 1 | Année 2 |
|---|---:|---:|
| Total forecasting examples | 1,826,182 | 1,752,879 |
| Complete lag examples | 1,825,206 | 1,752,092 |
| Incomplete lag examples | 976 | 787 |
| Feature completeness | 99.95% | 99.96% |

#### Key Findings

1. Historical lag features were successfully generated for both annual datasets.
2. More than 99.9% of forecasting examples contain all six historical lag features.
3. Missing historical measurements are limited and may reflect unavailable observations within the specified matching tolerance.
4. The 300-second lag exhibits slightly lower availability than the shorter historical lags.
5. No forecasting examples have been removed at this stage.

#### Interpretation

The high feature completeness indicates that timestamp-aware lag construction is suitable for the satellite power dataset.

The remaining incomplete examples will be considered after rolling statistical feature construction.

This approach preserves the maximum amount of available information while ensuring that the final machine learning dataset can be validated consistently.



### 2.5.6 Timestamp-Aware Rolling Statistical Feature Construction

Rolling statistical features summarize recent satellite power behavior using actual observation timestamps.

Two historical windows are considered:

- 120 seconds (2 minutes)
- 300 seconds (5 minutes)

For each window and each selected power variable (`NPWD2721` and `NPWD2802`), the following features are calculated:

1. **Rolling mean:** Represents the average power measurement within the historical window.
2. **Rolling standard deviation:** Represents variability in the power measurements within the historical window.

The calculations use only observations available at or before the prediction timestamp.

A minimum of two historical observations is required to calculate each statistic.

Rolling features are constructed independently for each annual dataset, preserving the original forecasting examples and previously generated lag features.

The resulting features will be validated before final dataset preparation.


In [19]:

# Section 2.5.6 — Timestamp-aware rolling statistical features

rolling_windows = [120, 300]
power_variables = ["NPWD2721", "NPWD2802"]

rolling_feature_data = {}

for year, features_df in lag_feature_data.items():

    # Preserve existing forecasting and lag features
    engineered_df = features_df.copy()

    # Use the corresponding original power observations
    history_df = (
        power_data[year][["ut_ms"] + power_variables]
        .sort_values("ut_ms")
        .copy()
    )

    # Convert timestamps to datetime for time-based rolling
    history_df["timestamp"] = pd.to_datetime(
        history_df["ut_ms"],
        unit="ms",
        utc=True
    )

    history_df = history_df.set_index("timestamp")

    # Construct rolling features for each historical window
    for window_sec in rolling_windows:

        rolling_stats = history_df[
            power_variables
        ].rolling(
            window=f"{window_sec}s",
            min_periods=2
        ).agg(["mean", "std"])

        rolling_stats.columns = [
            f"{variable}_rolling_{window_sec}s_{stat}"
            for variable, stat in rolling_stats.columns
        ]

        # Restore timestamp column for alignment
        rolling_stats = (
            rolling_stats.reset_index()
            .sort_values("timestamp")
        )

        # Match rolling statistics to forecasting timestamps
        prediction_times = engineered_df[
            ["ut_ms"]
        ].copy()

        prediction_times["timestamp"] = pd.to_datetime(
            prediction_times["ut_ms"],
            unit="ms",
            utc=True
        )

        matched_stats = pd.merge_asof(
            prediction_times.sort_values("timestamp"),
            rolling_stats,
            on="timestamp",
            direction="backward"
        )

        # Restore original forecasting row order
        matched_stats = matched_stats.sort_index()

        new_columns = [
            col for col in rolling_stats.columns
            if col != "timestamp"
        ]

        for column in new_columns:
            engineered_df[column] = (
                matched_stats[column].to_numpy()
            )

    rolling_feature_data[year] = engineered_df

    print(
        f"{year}: {len(engineered_df):,} examples, "
        f"{len(engineered_df.columns)} columns"
    )


annee1: 1,826,182 examples, 20 columns
annee2: 1,752,879 examples, 20 columns



### 2.5.7 Rolling Feature Validation

This section evaluates the completeness and numerical validity of the eight timestamp-aware rolling statistical features.

The validation examines:

- Missing values in each rolling feature.
- Non-finite values, including positive and negative infinity.
- Percentage of forecasting examples with valid rolling features.
- Overall completeness of the engineered feature dataset.

Missing values may occur when fewer than two historical observations are available within a rolling window.

The objective is to determine whether the rolling features are suitable for subsequent machine learning preparation without introducing data leakage.


In [20]:

# Section 2.5.7 — Rolling feature validation

rolling_validation_results = []

for year, df in rolling_feature_data.items():

    rolling_columns = [
        col for col in df.columns
        if "_rolling_" in col
    ]

    for column in rolling_columns:

        values = pd.to_numeric(
            df[column],
            errors="coerce"
        )

        finite_mask = np.isfinite(values.to_numpy())

        rolling_validation_results.append({
            "Year": year,
            "Feature": column,
            "Total Examples": len(df),
            "Missing Values": int(values.isna().sum()),
            "Non-finite Values": int((~finite_mask).sum()),
            "Valid (%)": round(
                100 * finite_mask.sum() / len(df), 4
            )
        })

rolling_validation_df = pd.DataFrame(
    rolling_validation_results
)

display(rolling_validation_df)


,Year,Feature,Total Examples,Missing Values,Non-finite Values,Valid (%)
0,annee1,NPWD2721_rolling_120s_mean,1826182,70,70,99.9962
1,annee1,NPWD2721_rolling_120s_std,1826182,70,70,99.9962
2,annee1,NPWD2802_rolling_120s_mean,1826182,70,70,99.9962
3,annee1,NPWD2802_rolling_120s_std,1826182,70,70,99.9962
4,annee1,NPWD2721_rolling_300s_mean,1826182,22,22,99.9988
5,annee1,NPWD2721_rolling_300s_std,1826182,22,22,99.9988
6,annee1,NPWD2802_rolling_300s_mean,1826182,22,22,99.9988
7,annee1,NPWD2802_rolling_300s_std,1826182,22,22,99.9988
8,annee2,NPWD2721_rolling_120s_mean,1752879,42,42,99.9976
9,annee2,NPWD2721_rolling_120s_std,1752879,42,42,99.9976



### 2.5.8 Final Engineered Feature Cleaning

The rolling feature validation identified a small number of missing values in the 120-second and 300-second statistical features.

For the initial forecasting dataset, examples containing missing or non-finite numerical features will be excluded rather than imputed.

This approach preserves the observed historical values, avoids introducing artificial measurements, and produces complete feature vectors for machine learning.

The original engineered datasets remain unchanged. A separate cleaned dataset is created for subsequent processing.


In [21]:

# Section 2.5.8 — Final engineered feature cleaning

cleaned_feature_data = {}
cleaning_results = []

for year, df in rolling_feature_data.items():

    cleaned_df = df.copy()

    # Identify numerical feature columns
    numeric_columns = cleaned_df.select_dtypes(
        include=[np.number]
    ).columns.tolist()

    # Replace infinite values with missing values
    cleaned_df[numeric_columns] = cleaned_df[
        numeric_columns
    ].replace([np.inf, -np.inf], np.nan)

    # Remove examples with incomplete numerical features
    cleaned_df = cleaned_df.dropna(
        subset=numeric_columns
    ).copy()

    cleaned_feature_data[year] = cleaned_df

    cleaning_results.append({
        "Year": year,
        "Before Cleaning": len(df),
        "After Cleaning": len(cleaned_df),
        "Removed": len(df) - len(cleaned_df),
        "Retention (%)": round(
            100 * len(cleaned_df) / len(df), 4
        )
    })

feature_cleaning_summary = pd.DataFrame(
    cleaning_results
)

display(feature_cleaning_summary)


,Year,Before Cleaning,After Cleaning,Removed,Retention (%)
0,annee1,1826182,1825206,976,99.9466
1,annee2,1752879,1752092,787,99.9551



## 2.6 Chronological Train, Validation and Test Splitting

Satellite power consumption is a time-dependent process. Therefore, the forecasting dataset must be divided chronologically rather than randomly.

The proposed data allocation is:

- Training set: 70%
- Validation set: 15%
- Test set: 15%

Chronological splitting preserves the temporal ordering of observations and supports a realistic evaluation of future power predictions.

Special attention will be given to the five-minute prediction horizon to prevent information leakage across dataset boundaries.

Feature scaling and model fitting will be performed only after the data partitions have been established.


In [22]:

# Section 2.6.1 — Inspect forecasting dataset structure

split_preparation_results = []

for year, df in cleaned_feature_data.items():

    print(f"\n{'=' * 65}")
    print(f"Dataset: {year}")
    print(f"{'=' * 65}")

    print("\nAvailable columns:")
    print(df.columns.tolist())

    print("\nFirst three observations:")
    display(df.head(3))

    print("\nLast three observations:")
    display(df.tail(3))

    # Identify timestamp-related columns
    timestamp_columns = [
        col for col in df.columns
        if any(
            keyword in col.lower()
            for keyword in ["time", "timestamp", "date"]
        )
    ]

    print("\nTimestamp-related columns:")
    print(timestamp_columns)

    split_preparation_results.append({
        "Year": year,
        "Observations": len(df),
        "Columns": len(df.columns),
        "Timestamp Columns": timestamp_columns
    })

split_preparation_summary = pd.DataFrame(
    split_preparation_results
)

display(split_preparation_summary)



Dataset: annee1

Available columns:
['ut_ms', 'NPWD2721', 'NPWD2802', 'target_ut_ms', 'target_NPWD2721', 'forecast_interval_sec', 'NPWD2721_lag_60s', 'NPWD2802_lag_60s', 'NPWD2721_lag_120s', 'NPWD2802_lag_120s', 'NPWD2721_lag_300s', 'NPWD2802_lag_300s', 'NPWD2721_rolling_120s_mean', 'NPWD2721_rolling_120s_std', 'NPWD2802_rolling_120s_mean', 'NPWD2802_rolling_120s_std', 'NPWD2721_rolling_300s_mean', 'NPWD2721_rolling_300s_std', 'NPWD2802_rolling_300s_mean', 'NPWD2802_rolling_300s_std']

First three observations:


,ut_ms,NPWD2721,NPWD2802,target_ut_ms,target_NPWD2721,forecast_interval_sec,NPWD2721_lag_60s,NPWD2802_lag_60s,NPWD2721_lag_120s,NPWD2802_lag_120s,NPWD2721_lag_300s,NPWD2802_lag_300s,NPWD2721_rolling_120s_mean,NPWD2721_rolling_120s_std,NPWD2802_rolling_120s_mean,NPWD2802_rolling_120s_std,NPWD2721_rolling_300s_mean,NPWD2721_rolling_300s_std,NPWD2802_rolling_300s_mean,NPWD2802_rolling_300s_std
10,1219363529558,0.159668,0.161594,1219363817537,0.159668,287.979,0.159668,0.156296,0.164957,0.161594,0.164957,0.156296,0.160990,0.002644,0.158945,0.003059,0.161784,0.002731,0.159475,0.002736
11,1219363561555,0.159668,0.161594,1219363849542,0.159668,287.987,0.159668,0.156296,0.164957,0.161594,0.159668,0.156296,0.159668,0.000000,0.158945,0.003059,0.161784,0.002731,0.160005,0.002559
12,1219363593572,0.164957,0.161594,1219363881537,0.159668,287.965,0.159668,0.161594,0.159668,0.156296,0.159668,0.161594,0.160990,0.002644,0.160269,0.002649,0.162312,0.002788,0.160005,0.002559



Last three observations:


,ut_ms,NPWD2721,NPWD2802,target_ut_ms,target_NPWD2721,forecast_interval_sec,NPWD2721_lag_60s,NPWD2802_lag_60s,NPWD2721_lag_120s,NPWD2802_lag_120s,NPWD2721_lag_300s,NPWD2802_lag_300s,NPWD2721_rolling_120s_mean,NPWD2721_rolling_120s_std,NPWD2802_rolling_120s_mean,NPWD2802_rolling_120s_std,NPWD2721_rolling_300s_mean,NPWD2721_rolling_300s_std,NPWD2802_rolling_300s_mean,NPWD2802_rolling_300s_std
1826179,1278719624561,0.159668,0.161594,1278719912556,0.159668,287.995,0.164957,0.161594,0.159668,0.161594,0.159668,0.156296,0.160990,0.002644,0.16027,0.002649,0.160726,0.002230,0.160534,0.002234
1826180,1278719656556,0.159668,0.161594,1278719944561,0.159668,288.005,0.159668,0.156296,0.159668,0.161594,0.159668,0.161594,0.160990,0.002644,0.16027,0.002649,0.160726,0.002230,0.160534,0.002234
1826181,1278719688564,0.159668,0.161594,1278719976556,0.164957,287.992,0.159668,0.161594,0.164957,0.161594,0.164957,0.161594,0.159668,0.000000,0.16027,0.002649,0.160197,0.001673,0.160534,0.002234



Timestamp-related columns:
[]

Dataset: annee2

Available columns:
['ut_ms', 'NPWD2721', 'NPWD2802', 'target_ut_ms', 'target_NPWD2721', 'forecast_interval_sec', 'NPWD2721_lag_60s', 'NPWD2802_lag_60s', 'NPWD2721_lag_120s', 'NPWD2802_lag_120s', 'NPWD2721_lag_300s', 'NPWD2802_lag_300s', 'NPWD2721_rolling_120s_mean', 'NPWD2721_rolling_120s_std', 'NPWD2802_rolling_120s_mean', 'NPWD2802_rolling_120s_std', 'NPWD2721_rolling_300s_mean', 'NPWD2721_rolling_300s_std', 'NPWD2802_rolling_300s_mean', 'NPWD2802_rolling_300s_std']

First three observations:


,ut_ms,NPWD2721,NPWD2802,target_ut_ms,target_NPWD2721,forecast_interval_sec,NPWD2721_lag_60s,NPWD2802_lag_60s,NPWD2721_lag_120s,NPWD2802_lag_120s,NPWD2721_lag_300s,NPWD2802_lag_300s,NPWD2721_rolling_120s_mean,NPWD2721_rolling_120s_std,NPWD2802_rolling_120s_mean,NPWD2802_rolling_120s_std,NPWD2721_rolling_300s_mean,NPWD2721_rolling_300s_std,NPWD2802_rolling_300s_mean,NPWD2802_rolling_300s_std
9,1278720328564,0.164957,0.161594,1278720616559,0.159668,287.995,0.159668,0.161594,0.159668,0.156296,0.159668,0.161594,0.160990,0.002644,0.160269,0.002649,0.160256,0.001763,0.160417,0.002336
11,1278720392564,0.164957,0.161594,1278720680559,0.159668,287.995,0.164957,0.161594,0.159668,0.161594,0.159668,0.161594,0.163635,0.002644,0.160269,0.002649,0.161255,0.002555,0.160534,0.002234
12,1278720424559,0.159668,0.156296,1278720712564,0.159668,288.005,0.164957,0.161594,0.159668,0.156296,0.159668,0.161594,0.163635,0.002644,0.160269,0.002649,0.161255,0.002555,0.160005,0.002559



Last three observations:


,ut_ms,NPWD2721,NPWD2802,target_ut_ms,target_NPWD2721,forecast_interval_sec,NPWD2721_lag_60s,NPWD2802_lag_60s,NPWD2721_lag_120s,NPWD2802_lag_120s,NPWD2721_lag_300s,NPWD2802_lag_300s,NPWD2721_rolling_120s_mean,NPWD2721_rolling_120s_std,NPWD2802_rolling_120s_mean,NPWD2802_rolling_120s_std,NPWD2721_rolling_300s_mean,NPWD2721_rolling_300s_std,NPWD2802_rolling_300s_mean,NPWD2802_rolling_300s_std
1752876,1338076425758,0.159668,0.161594,1338076713753,0.159668,287.995,0.164957,0.161594,0.159668,0.156296,0.164957,0.156296,0.163635,0.002645,0.161594,0.000000,0.162841,0.002731,0.159475,0.002736
1752877,1338076457753,0.159668,0.156296,1338076745758,0.159668,288.005,0.164957,0.161594,0.164957,0.161594,0.159668,0.156296,0.162312,0.003054,0.160270,0.002649,0.162841,0.002731,0.159475,0.002736
1752878,1338076489758,0.164957,0.161594,1338076777753,0.159668,287.995,0.159668,0.161594,0.164957,0.161594,0.164957,0.161594,0.162313,0.003054,0.160270,0.002649,0.162841,0.002731,0.159475,0.002736



Timestamp-related columns:
[]


,Year,Observations,Columns,Timestamp Columns
0,annee1,1825206,20,[]
1,annee2,1752092,20,[]



## 2.6.2 — Chronological Train, Validation, and Test Split

The cleaned forecasting datasets are divided into training (70%), validation (15%), and testing (15%) partitions.

Unlike random splitting, chronological splitting preserves the temporal order of satellite telemetry observations. This ensures that model training uses earlier observations while validation and testing evaluate predictions on later periods.

The reference timestamp (`ut_ms`) determines the chronological partition. The future timestamp (`target_ut_ms`) is checked to prevent target leakage across partition boundaries.

Each year is partitioned separately, preserving its original temporal structure.


In [23]:

# Section 2.6.2 — Leakage-safe chronological splitting

train_data = {}
validation_data = {}
test_data = {}

split_summary = []

for year, df in cleaned_feature_data.items():

    # Preserve chronological order
    df = df.sort_values("ut_ms").reset_index(drop=True)

    n = len(df)

    train_end = int(n * 0.70)
    validation_end = int(n * 0.85)

    # Define boundaries using reference timestamps
    validation_start_time = df["ut_ms"].iloc[train_end]
    test_start_time = df["ut_ms"].iloc[validation_end]

    # Assign samples chronologically and prevent future-target leakage
    train_df = df.iloc[:train_end].loc[
        lambda x: x["target_ut_ms"] < validation_start_time
    ].copy()

    validation_df = df.iloc[train_end:validation_end].loc[
        lambda x: x["target_ut_ms"] < test_start_time
    ].copy()

    test_df = df.iloc[validation_end:].copy()

    train_data[year] = train_df
    validation_data[year] = validation_df
    test_data[year] = test_df

    split_summary.append({
        "Year": year,
        "Training": len(train_df),
        "Validation": len(validation_df),
        "Testing": len(test_df),
        "Excluded at Boundaries": (
            n - len(train_df) - len(validation_df) - len(test_df)
        ),
        "Total Retained": (
            len(train_df) + len(validation_df) + len(test_df)
        )
    })

chronological_split_summary = pd.DataFrame(split_summary)

display(chronological_split_summary)


,Year,Training,Validation,Testing,Excluded at Boundaries,Total Retained
0,annee1,1277635,273772,273781,18,1825188
1,annee2,1226455,262805,262814,18,1752074



### 2.6.3 — Validation of Chronological Data Splitting

The chronological partitions are checked to ensure that the forecasting datasets remain temporally separated.

For each dataset, the following conditions must hold:

1. Training reference timestamps precede validation reference timestamps.
2. Validation reference timestamps precede testing reference timestamps.
3. Training target timestamps precede the start of validation.
4. Validation target timestamps precede the start of testing.

These checks help prevent future-target leakage and support reliable evaluation of the five-minute-ahead forecasting model.


In [24]:

# Section 2.6.3 — Validate chronological separation

temporal_split_results = []

for year in train_data:

    train_df = train_data[year]
    val_df = validation_data[year]
    test_df = test_data[year]

    temporal_split_results.append({
        "Year": year,

        "Train Before Validation": bool(
            train_df["ut_ms"].max() <
            val_df["ut_ms"].min()
        ),

        "Validation Before Test": bool(
            val_df["ut_ms"].max() <
            test_df["ut_ms"].min()
        ),

        "No Train Target Leakage": bool(
            train_df["target_ut_ms"].max() <
            val_df["ut_ms"].min()
        ),

        "No Validation Target Leakage": bool(
            val_df["target_ut_ms"].max() <
            test_df["ut_ms"].min()
        )
    })

temporal_split_validation = pd.DataFrame(
    temporal_split_results
)

display(temporal_split_validation)

assert temporal_split_validation.drop(
    columns="Year"
).to_numpy().all(), "Temporal separation validation failed."


,Year,Train Before Validation,Validation Before Test,No Train Target Leakage,No Validation Target Leakage
0,annee1,True,True,True,True
1,annee2,True,True,True,True



## 2.7 — Feature Scaling

Feature scaling standardizes numerical input variables so that differences in measurement ranges do not disproportionately influence model training.

StandardScaler is used to transform the features to approximately zero mean and unit variance based on the training dataset.

To prevent data leakage:

1. Scaling parameters are learned exclusively from training data.
2. The fitted scaler is applied unchanged to validation and testing data.
3. Forecast targets are retained in their original units.
4. Timestamp and forecast metadata columns are excluded from scaling.

Each year's dataset is scaled separately using its corresponding training partition.


In [25]:

# Section 2.7.1 — Identify model input features

metadata_columns = [
    "ut_ms",
    "target_ut_ms",
    "target_NPWD2721",
    "forecast_interval_sec"
]

feature_columns = [
    col
    for col in train_data["annee1"].columns
    if col not in metadata_columns
]

# Confirm identical feature structure across datasets
for year in train_data:
    assert all(
        col in train_data[year].columns
        for col in feature_columns
    ), f"Missing input features in {year}"

print("Number of input features:", len(feature_columns))

print("\nSelected model input features:")
for i, feature in enumerate(feature_columns, start=1):
    print(f"{i}. {feature}")

print("\nExcluded metadata columns:")
for column in metadata_columns:
    print("-", column)


Number of input features: 16

Selected model input features:
1. NPWD2721
2. NPWD2802
3. NPWD2721_lag_60s
4. NPWD2802_lag_60s
5. NPWD2721_lag_120s
6. NPWD2802_lag_120s
7. NPWD2721_lag_300s
8. NPWD2802_lag_300s
9. NPWD2721_rolling_120s_mean
10. NPWD2721_rolling_120s_std
11. NPWD2802_rolling_120s_mean
12. NPWD2802_rolling_120s_std
13. NPWD2721_rolling_300s_mean
14. NPWD2721_rolling_300s_std
15. NPWD2802_rolling_300s_mean
16. NPWD2802_rolling_300s_std

Excluded metadata columns:
- ut_ms
- target_ut_ms
- target_NPWD2721
- forecast_interval_sec



### 2.7.2 — Standardization of Model Input Features

The 16 selected numerical input features are standardized using `StandardScaler` from scikit-learn.

For each yearly dataset, the scaler is fitted exclusively on the training partition. The fitted transformation is then applied unchanged to the validation and testing partitions.

This approach prevents information from the validation or testing periods from influencing the scaling parameters.

The forecasting target (`target_NPWD2721`) and timestamp-related metadata remain unchanged.


In [26]:

# Section 2.7.2 — Fit and apply StandardScaler

from sklearn.preprocessing import StandardScaler

scalers = {}

scaled_train_data = {}
scaled_validation_data = {}
scaled_test_data = {}

for year in train_data:

    # Initialize the scaler
    scaler = StandardScaler()

    # Fit using training features only
    scaler.fit(train_data[year][feature_columns])

    # Preserve original DataFrame structure
    train_scaled = train_data[year].copy()
    validation_scaled = validation_data[year].copy()
    test_scaled = test_data[year].copy()

    # Apply the training-fitted scaler
    train_scaled[feature_columns] = scaler.transform(
        train_data[year][feature_columns]
    )

    validation_scaled[feature_columns] = scaler.transform(
        validation_data[year][feature_columns]
    )

    test_scaled[feature_columns] = scaler.transform(
        test_data[year][feature_columns]
    )

    # Store fitted scaler and transformed datasets
    scalers[year] = scaler

    scaled_train_data[year] = train_scaled
    scaled_validation_data[year] = validation_scaled
    scaled_test_data[year] = test_scaled

    print(
        f"{year}: "
        f"Train={train_scaled.shape}, "
        f"Validation={validation_scaled.shape}, "
        f"Test={test_scaled.shape}"
    )


annee1: Train=(1277635, 20), Validation=(273772, 20), Test=(273781, 20)
annee2: Train=(1226455, 20), Validation=(262805, 20), Test=(262814, 20)



### 2.7.3 — Validation of Feature Scaling

The standardized datasets are validated to confirm that feature scaling was applied correctly.

The validation checks whether training features have approximately zero mean and unit standard deviation, except for constant features.

Additionally, the procedure verifies that all scaled feature values are finite and that the original prediction targets and timestamp-related metadata remain unchanged.

These checks ensure that the preprocessing pipeline produces consistent numerical inputs for subsequent machine learning experiments.


In [27]:

# Section 2.7.3 — Validate feature scaling

import numpy as np

scaling_validation_results = []

for year in train_data:

    original_train = train_data[year]
    scaled_train = scaled_train_data[year]

    scaler = scalers[year]

    # StandardScaler uses population variance (ddof=0)
    scaled_means = scaled_train[feature_columns].mean()
    scaled_stds = scaled_train[feature_columns].std(ddof=0)

    # Constant features have zero variance
    nonconstant = scaler.var_ > 0

    mean_valid = np.allclose(
        scaled_means.to_numpy(),
        0,
        atol=1e-6
    )

    std_valid = np.allclose(
        scaled_stds.to_numpy()[nonconstant],
        1,
        atol=1e-6
    )

    all_finite = all(
        np.isfinite(
            partition[feature_columns].to_numpy()
        ).all()
        for partition in [
            scaled_train_data[year],
            scaled_validation_data[year],
            scaled_test_data[year]
        ]
    )

    # Confirm metadata and targets are unchanged
    metadata_unchanged = all(
        original[metadata_columns].equals(
            scaled[metadata_columns]
        )
        for original, scaled in [
            (train_data[year], scaled_train_data[year]),
            (validation_data[year], scaled_validation_data[year]),
            (test_data[year], scaled_test_data[year])
        ]
    )

    scaling_validation_results.append({
        "Year": year,
        "Training Mean ≈ 0": bool(mean_valid),
        "Training Std ≈ 1": bool(std_valid),
        "All Features Finite": bool(all_finite),
        "Metadata Unchanged": bool(metadata_unchanged),
        "Constant Features": int((~nonconstant).sum())
    })

scaling_validation_df = pd.DataFrame(
    scaling_validation_results
)

display(scaling_validation_df)

# Stop if any required validation fails
validation_columns = [
    "Training Mean ≈ 0",
    "Training Std ≈ 1",
    "All Features Finite",
    "Metadata Unchanged"
]

assert scaling_validation_df[
    validation_columns
].to_numpy().all(), "Feature scaling validation failed."


,Year,Training Mean ≈ 0,Training Std ≈ 1,All Features Finite,Metadata Unchanged,Constant Features
0,annee1,True,True,True,True,0
1,annee2,True,True,True,True,0



## 2.8 — Preparing the Final Model-Ready Datasets

This section organizes the validated and standardized satellite power data into input features (`X`) and prediction targets (`y`).

The prediction objective is to estimate the future value of `NPWD2721` approximately five minutes ahead using the current measurements, historical lag features, and rolling statistical features.

The previously established chronological training, validation, and testing partitions are preserved to prevent temporal data leakage.

The target variable is kept in its original numerical scale so that future model predictions can be evaluated using physically interpretable error metrics.


In [28]:

# Section 2.8.1 — Separate model inputs and prediction targets

model_ready_data = {}

target_column = "target_NPWD2721"

for year in scaled_train_data:

    model_ready_data[year] = {}

    for split_name, scaled_dataset in [
        ("train", scaled_train_data[year]),
        ("validation", scaled_validation_data[year]),
        ("test", scaled_test_data[year])
    ]:

        X = scaled_dataset[feature_columns].to_numpy(
            dtype=np.float32
        )

        y = scaled_dataset[target_column].to_numpy(
            dtype=np.float32
        )

        model_ready_data[year][split_name] = {
            "X": X,
            "y": y
        }

        print(
            f"{year} | {split_name}: "
            f"X={X.shape}, y={y.shape}"
        )


annee1 | train: X=(1277635, 16), y=(1277635,)
annee1 | validation: X=(273772, 16), y=(273772,)
annee1 | test: X=(273781, 16), y=(273781,)
annee2 | train: X=(1226455, 16), y=(1226455,)
annee2 | validation: X=(262805, 16), y=(262805,)
annee2 | test: X=(262814, 16), y=(262814,)



### 2.8.2 — Final Validation of Model-Ready Arrays

Before exporting the processed datasets, the model-ready arrays are validated to ensure consistency and numerical reliability.

The validation checks confirm that input features and prediction targets have matching observation counts, that each input contains the expected 16 features, and that all values are finite.

The numerical data type is also verified to ensure memory-efficient storage and compatibility with machine learning algorithms.

This final validation establishes that the training, validation, and testing arrays are ready for subsequent model development.


In [29]:

# Section 2.8.2 — Final validation of model-ready arrays

validation_results = []

for year, splits in model_ready_data.items():

    for split_name, arrays in splits.items():

        X = arrays["X"]
        y = arrays["y"]

        checks = {
            "Year": year,
            "Dataset": split_name,
            "Observations": len(X),
            "Input Features": X.shape[1],
            "Matching X and y": len(X) == len(y),
            "16 Features": X.shape[1] == 16,
            "Finite Inputs": bool(np.isfinite(X).all()),
            "Finite Targets": bool(np.isfinite(y).all()),
            "Float32": X.dtype == np.float32 and y.dtype == np.float32
        }

        validation_results.append(checks)

model_ready_validation = pd.DataFrame(validation_results)

display(model_ready_validation)

# Confirm all required checks passed
check_columns = [
    "Matching X and y",
    "16 Features",
    "Finite Inputs",
    "Finite Targets",
    "Float32"
]

assert model_ready_validation[check_columns].to_numpy().all(), (
    "Model-ready dataset validation failed."
)

print("All model-ready datasets passed final validation.")


,Year,Dataset,Observations,Input Features,Matching X and y,16 Features,Finite Inputs,Finite Targets,Float32
0,annee1,train,1277635,16,True,True,True,True,True
1,annee1,validation,273772,16,True,True,True,True,True
2,annee1,test,273781,16,True,True,True,True,True
3,annee2,train,1226455,16,True,True,True,True,True
4,annee2,validation,262805,16,True,True,True,True,True
5,annee2,test,262814,16,True,True,True,True,True


All model-ready datasets passed final validation.



## 2.9 — Exporting Model-Ready Datasets

The validated model-ready datasets are exported for reproducible machine learning experiments.

Each dataset contains 16 standardized input features and the corresponding unscaled satellite power prediction target (`target_NPWD2721`).

Training, validation, and testing datasets are stored separately for each year, preserving the chronological partitions established during preprocessing.

The fitted feature scalers and feature definitions are also saved to ensure that future model development and inference use consistent transformations.

These exported artifacts allow subsequent notebooks to load the prepared data directly without repeating the preprocessing pipeline.


In [30]:

# Section 2.9.1 — Export model-ready datasets and preprocessing artifacts

from pathlib import Path
import joblib

# Save artifacts inside the project's data directory
output_dir = Path.cwd().resolve().parent / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# Export the six model-ready datasets
for year, splits in model_ready_data.items():

    for split_name, arrays in splits.items():

        output_path = output_dir / f"{year}_{split_name}.npz"

        np.savez_compressed(
            output_path,
            X=arrays["X"],
            y=arrays["y"]
        )

        print(f"Saved: {output_path.name}")

# Save fitted scalers and feature definitions
joblib.dump(
    scalers,
    output_dir / "feature_scalers.joblib"
)

joblib.dump(
    {
        "feature_columns": feature_columns,
        "target_column": target_column
    },
    output_dir / "feature_metadata.joblib"
)

print("\nAll preprocessing artifacts exported successfully.")
print(f"Output directory: {output_dir}")


Saved: annee1_train.npz
Saved: annee1_validation.npz
Saved: annee1_test.npz
Saved: annee2_train.npz
Saved: annee2_validation.npz
Saved: annee2_test.npz

All preprocessing artifacts exported successfully.
Output directory: C:\Users\DAMISA\Desktop\DAMISA\ECE M2\M2 Sem 1\FABLAB\Dream Orbit\data\processed



### 2.9.2 — Verification of Exported Preprocessing Artifacts

This section verifies that the exported datasets and preprocessing metadata were saved successfully.

The verification confirms the existence of all expected files and checks that the saved feature and target arrays can be reloaded with the correct dimensions.

This ensures that subsequent machine learning notebooks can reuse the processed data without repeating the preprocessing workflow.


In [31]:

# Section 2.9.2 — Verify exported preprocessing artifacts

expected_files = [
    f"{year}_{split}.npz"
    for year in model_ready_data
    for split in ("train", "validation", "test")
] + [
    "feature_scalers.joblib",
    "feature_metadata.joblib"
]

print("Export directory:", output_dir)
print("\nFile verification:")

for filename in expected_files:
    file_path = output_dir / filename
    assert file_path.is_file(), f"Missing file: {filename}"
    print(f"PASS: {filename}")

print("\nDataset reload verification:")

for year, splits in model_ready_data.items():
    for split_name, original in splits.items():

        file_path = output_dir / f"{year}_{split_name}.npz"

        with np.load(file_path) as saved:
            X = saved["X"]
            y = saved["y"]

            assert X.shape == original["X"].shape
            assert y.shape == original["y"].shape
            assert np.array_equal(X, original["X"])
            assert np.array_equal(y, original["y"])

        print(f"PASS: {year} | {split_name}")

# Verify saved preprocessing metadata
saved_metadata = joblib.load(
    output_dir / "feature_metadata.joblib"
)

saved_scalers = joblib.load(
    output_dir / "feature_scalers.joblib"
)

assert list(saved_metadata["feature_columns"]) == list(feature_columns)
assert saved_metadata["target_column"] == target_column
assert set(saved_scalers) == set(scalers)

print("\nAll exported preprocessing artifacts verified successfully.")


Export directory: C:\Users\DAMISA\Desktop\DAMISA\ECE M2\M2 Sem 1\FABLAB\Dream Orbit\data\processed

File verification:
PASS: annee1_train.npz
PASS: annee1_validation.npz
PASS: annee1_test.npz
PASS: annee2_train.npz
PASS: annee2_validation.npz
PASS: annee2_test.npz
PASS: feature_scalers.joblib
PASS: feature_metadata.joblib

Dataset reload verification:
PASS: annee1 | train
PASS: annee1 | validation
PASS: annee1 | test
PASS: annee2 | train
PASS: annee2 | validation
PASS: annee2 | test

All exported preprocessing artifacts verified successfully.


In [33]:
# Section 2.9.3 — Verify Git ignores processed datasets

import subprocess
from pathlib import Path

# Identify the Git repository root
repo_root = Path.cwd().resolve()

# Verify that processed datasets are excluded from Git
result = subprocess.run(
    ["git", "check-ignore", "data/processed/annee1_train.npz"],
    cwd=repo_root,
    capture_output=True,
    text=True
)

print("Git ignore verification:")

if result.returncode == 0:
    print("PASS: Processed datasets are excluded from Git.")
    print("Ignored path:", result.stdout.strip())
else:
    print("FAIL: Processed datasets are not excluded from Git.")
    print(result.stderr.strip())

Git ignore verification:
PASS: Processed datasets are excluded from Git.
Ignored path: data/processed/annee1_train.npz
